# 03 - Tổng hợp bảng biểu phục vụ báo cáo
Notebook này gom các bảng và chỉ số cuối cùng từ dữ liệu đã làm sạch để chèn trực tiếp vào báo cáo Word.

In [1]:
import os
import pandas as pd

In [2]:
os.makedirs("outputs/report", exist_ok=True)

df = pd.read_csv("data/cleaned_reviews.csv")
summary = pd.read_csv("outputs/eda_summary.csv")

display(df.head())
display(summary)

,review_id,shop_id,item_id,user_id,rating_star,comment,created_at,like_count,product_items,source,comment_clean,review_len
0,Anh KhoiPD DX_201234,FPTShop,925801,Anh KhoiPD DX,4.0,Máy dùng tốt,NaN,0,NaN,FPTShop,máy dùng tốt,12
1,chi mai_199345,FPTShop,925128,chi mai,5.0,siêu dth,NaN,0,NaN,FPTShop,siêu dth,8
2,Trần Minh Huy_185078,FPTShop,922302,Trần Minh Huy,5.0,"Con máy này chạy mượt mà cực kỳ, chơi game hay...",NaN,0,NaN,FPTShop,con máy này chạy mượt mà cực kỳ chơi game hay ...,91
3,Phan Văn Hải_184943,FPTShop,922302,Phan Văn Hải,4.0,"E này thiết kế đẹp mắt lắm mọi người ạ, màu đe...",NaN,0,NaN,FPTShop,e này thiết kế đẹp mắt lắm mọi người ạ màu đen...,91
4,Huỳnh Thảo Vy_183464,FPTShop,922302,Huỳnh Thảo Vy,4.0,"Con máy đáng đồng tiền bát gạo, mọi người cứ y...",NaN,0,NaN,FPTShop,con máy đáng đồng tiền bát gạo mọi người cứ yê...,59


,metric,value
0,count_reviews,810.0000
1,min_rating,1.0000
2,max_rating,5.0000
3,avg_rating,4.8086
4,median_rating,5.0000
5,avg_review_len,117.7600
6,min_review_len,3.0000
7,max_review_len,400.0000


In [ ]:
if "has_reply" not in df.columns:
    if "reply_content" in df.columns:
        df["has_reply"] = df["reply_content"].fillna("").astype(str).str.strip().str.len() > 0
    else:
        df["has_reply"] = False

group_cols = ["shop_id", "item_id"]
for optional_col in ["product_name", "brand"]:
    if optional_col in df.columns:
        group_cols.append(optional_col)

table_by_item = df.groupby(group_cols).agg(
    review_count=("review_id", "count"),
    avg_rating=("rating_star", "mean"),
    avg_review_len=("review_len", "mean"),
    reply_count=("has_reply", "sum")
).reset_index()

table_by_item["avg_rating"] = table_by_item["avg_rating"].round(3)
table_by_item["avg_review_len"] = table_by_item["avg_review_len"].round(2)
table_by_item["reply_rate"] = (table_by_item["reply_count"] / table_by_item["review_count"]).round(4)

display(table_by_item.sort_values("review_count", ascending=False).head(20))
table_by_item.to_csv("outputs/report/table_by_item.csv", index=False, encoding="utf-8-sig")

,shop_id,item_id,review_count,avg_rating,avg_review_len
1,FPTShop,881861,6,5.000,114.83
2,FPTShop,881888,6,5.000,113.67
3,FPTShop,888284,6,4.667,56.50
9,FPTShop,904755,6,5.000,144.83
4,FPTShop,888669,6,4.667,57.17
8,FPTShop,904510,6,5.000,141.17
7,FPTShop,898378,6,5.000,115.00
10,FPTShop,904945,6,5.000,144.50
14,FPTShop,909183,6,4.000,75.17
13,FPTShop,906460,6,5.000,39.67


In [4]:
table_positive_negative = pd.DataFrame({
    "nhom": ["Tich_cuc_4_5_sao", "Tieu_cuc_1_2_sao", "Trung_tinh_3_sao"],
    "so_luong": [
        int((df["rating_star"] >= 4).sum()),
        int((df["rating_star"] <= 2).sum()),
        int((df["rating_star"] == 3).sum())
    ]
})

display(table_positive_negative)
table_positive_negative.to_csv("outputs/report/table_sentiment_groups.csv", index=False, encoding="utf-8-sig")

,nhom,so_luong
0,Tich_cuc_4_5_sao,799
1,Tieu_cuc_1_2_sao,7
2,Trung_tinh_3_sao,4


In [ ]:
table_data_dictionary = pd.DataFrame([
    ["review_id", "Mã review duy nhất"],
    ["shop_id", "Nguồn bán hoặc mã shop (FPTShop)"],
    ["item_id", "Mã content dùng để gọi API comment của FPTShop"],
    ["product_name", "Tên sản phẩm"],
    ["brand", "Hãng sản phẩm"],
    ["price", "Giá niêm yết hoặc giá tham chiếu"],
    ["final_price", "Giá bán cuối cùng (nếu có)"],
    ["rating_count_total", "Tổng số lượt đánh giá của sản phẩm"],
    ["user_id", "Mã hoặc định danh người dùng đánh giá"],
    ["rating_star", "Số sao đánh giá (1-5)"],
    ["review_title", "Tiêu đề review (nếu có)"],
    ["comment", "Nội dung review gốc"],
    ["comment_clean", "Nội dung review sau làm sạch"],
    ["verified_purchase", "Cờ xác nhận đã mua hàng (nếu có)"],
    ["image_product", "Ảnh đại diện sản phẩm"],
    ["image_review", "Ảnh đính kèm đánh giá (nếu có)"],
    ["created_at", "Thời gian tạo review"],
    ["like_count", "Số lượt hữu ích"],
    ["product_items", "Biến thể/thuộc tính sản phẩm trong review (nếu có)"],
    ["reply_content", "Nội dung phản hồi từ shop/quản trị (nếu có)"],
    ["reply_created_at", "Thời gian phản hồi"],
    ["reply_user_id", "Định danh người phản hồi"],
    ["reply_is_admin", "Cờ phản hồi từ quản trị viên"],
    ["reply_like_count", "Số lượt hữu ích của phản hồi"],
    ["has_reply", "Cờ review có phản hồi từ shop/quản trị"],
    ["review_len", "Độ dài ký tự của review"],
    ["source", "Nguồn dữ liệu (FPTShop)"],
], columns=["ten_cot", "y_nghia"])

display(table_data_dictionary)
table_data_dictionary.to_csv("outputs/report/table_data_dictionary.csv", index=False, encoding="utf-8-sig")

,ten_cot,y_nghia
0,review_id,Mã review duy nhất
1,shop_id,Nguồn bán hoặc mã shop (FPTShop)
2,item_id,Mã sản phẩm/UPC từ FPTShop
3,user_id,Mã hoặc định danh người dùng đánh giá
4,rating_star,Số sao đánh giá (1-5)
5,comment,Nội dung review gốc
6,comment_clean,Nội dung review sau làm sạch
7,created_at,Thời gian tạo review
8,like_count,Số lượt hữu ích
9,review_len,Độ dài ký tự của review


In [6]:
print("Đã tạo các bảng trong thư mục outputs/report")

Đã tạo các bảng trong thư mục outputs/report
